### 04 —  MLflow + MinIO 

In [2]:
import os, sys, time, glob, pathlib, subprocess, urllib.request

os.chdir("/kaggle/working")
REPO = "/kaggle/working/repo"
subprocess.run(["rm","-rf",REPO])
subprocess.run(["git","clone","-b","project-defense",
    "https://github.com/FilthyName/Processing_and_analysis_of_medical_images.git", REPO], check=True)
os.chdir(REPO); sys.path.append(f"{REPO}/src")

META = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)[0]
IMAGES_DIR = "/kaggle/working/images"; pathlib.Path(IMAGES_DIR).mkdir(exist_ok=True)
for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True):
    link = os.path.join(IMAGES_DIR, os.path.basename(p))
    if not os.path.exists(link): os.symlink(p, link)
print("картинок:", len(glob.glob(f"{IMAGES_DIR}/*.jpg")))

subprocess.run(["python","src/make_splits.py","--metadata",META,
    "--images-dir",IMAGES_DIR,"--out-dir","data/splits","--seed","42"], check=True)

subprocess.run(["pip","install","-q","mlflow==2.16.2","boto3","hydra-core","omegaconf"], check=True)

os.makedirs("/kaggle/working/minio-data", exist_ok=True)
for name,url in [("minio","https://dl.min.io/server/minio/release/linux-amd64/minio"),
                 ("mc","https://dl.min.io/client/mc/release/linux-amd64/mc")]:
    f=f"/kaggle/working/{name}"
    if not os.path.exists(f): urllib.request.urlretrieve(url,f); os.chmod(f,0o755)

os.environ.update(MINIO_ROOT_USER="minioadmin", MINIO_ROOT_PASSWORD="minioadmin")
subprocess.Popen(["/kaggle/working/minio","server","/kaggle/working/minio-data",
    "--address",":9000","--console-address",":9001"],
    stdout=open("/kaggle/working/minio.log","w"), stderr=subprocess.STDOUT)
time.sleep(8)
subprocess.run(["/kaggle/working/mc","alias","set","local","http://localhost:9000","minioadmin","minioadmin"])
subprocess.run(["/kaggle/working/mc","mb","-p","local/mlflow"])

os.environ.update(
    MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
    AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin",
    MLFLOW_TRACKING_URI="http://localhost:5000")

subprocess.Popen(["mlflow","server",
    "--backend-store-uri","sqlite:////kaggle/working/mlflow.db",
    "--artifacts-destination","s3://mlflow/","--serve-artifacts",
    "--host","0.0.0.0","--port","5000"],
    env=os.environ.copy(), stdout=open("/kaggle/working/mlflow.log","w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        if urllib.request.urlopen("http://localhost:5000/health").read(): break
    except Exception: time.sleep(2)
print("MLflow:", urllib.request.urlopen("http://localhost:5000/health").read())

Cloning into '/kaggle/working/repo'...


картинок: 10015


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 17.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0

MLflow: b'OK'


In [3]:
!cd /kaggle/working/repo && python src/dl_experiments.py \
  data.images_dir=/kaggle/working/images \
  data.splits_dir=/kaggle/working/repo/data/splits

seed: 42
device: auto
data:
  splits_dir: /kaggle/working/repo/data/splits
  images_dir: /kaggle/working/images
  image_size: 224
  num_workers: 2
  classes:
  - akiec
  - bcc
  - bkl
  - df
  - mel
  - nv
  - vasc
  normalize_mean:
  - 0.485
  - 0.456
  - 0.406
  normalize_std:
  - 0.229
  - 0.224
  - 0.225
model:
  arch: vit_b_16
  pretrained: true
  unfreeze_last_blocks: 2
  dropout: 0.3
  optimizer: adamw
  lr: 1.0e-05
  weight_decay: 0.0001
train:
  epochs: 10
  batch_size: 16
  augmentation: strong
  use_class_weights: true
  early_stopping_patience: 0
  selection_metric: macro_f1
baseline:
  enabled: true
  arch: resnet18
  epochs: 5
  lr: 0.001
  optimizer: adam
  augmentation: light
robustness:
  enabled: true
  sample_size: 300
  perturbations:
  - hflip
  - rotate10
  - brightness
  - gaussian_noise
  - jpeg_compression
error_analysis:
  num_examples: 16
mel_threshold:
  enabled: true
  positive: mel
  target_sensitivity: 0.8
mlflow:
  tracking_uri: http://localhost:5000
  e

In [4]:
import subprocess
subprocess.run(["pip","install","-q","mlflow==2.16.2","boto3"], check=True)

import os
os.environ.update(MLFLOW_TRACKING_URI="http://localhost:5000",
                  MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
                  AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin")
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
c = mlflow.tracking.MlflowClient()

print("=== PRD-модель ===")
for mv in c.search_model_versions("name='skin_lesion_vit_b16'"):
    print(f"  версия {mv.version} | теги: {mv.tags}")

print("\n=== Раны и метрики ===")
exp = c.get_experiment_by_name("skin_lesion_isic2018")
for r in c.search_runs(exp.experiment_id):
    m = r.data.metrics
    print(f"  {r.data.tags.get('mlflow.runName')}: "
          f"test_macro_f1={round(m.get('test_macro_f1',0),4)}, "
          f"mel_sens_thr={round(m.get('test_mel_sensitivity_thr',0),4)}")

=== PRD-модель ===
  версия 1 | теги: {'stage': 'PRD', 'mel_threshold': '0.14', 'mel_index': '4'}

=== Раны и метрики ===
  vit_b16_final: test_macro_f1=0.5801, mel_sens_thr=0.7877
  baseline_resnet18: test_macro_f1=0.4233, mel_sens_thr=0


In [5]:
!cd /kaggle/working/repo && python src/dl_demonstration.py \
  data.images_dir=/kaggle/working/images \
  data.splits_dir=/kaggle/working/repo/data/splits

PRD-модель загружена (skin_lesion_vit_b16@PRD) | порог mel=0.14 (idx 4)
demo.image не задан — беру первый пример из test: /kaggle/working/images/ISIC_0027419.jpg

Предсказание для ISIC_0027419.jpg:
  -> bkl  (p=0.852)
  top-k по вероятности:
     bkl      0.852
     mel      0.069
     akiec    0.052


In [7]:
import os
for f in ["/kaggle/working/checkpoint7_results.zip", "/kaggle/working/ckpt7_core.zip"]:
    if os.path.exists(f): os.remove(f)
# посмотреть, сколько занято
!df -h /kaggle/working
!du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -10

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.3G   19G   7% /kaggle/working
4.0K	/kaggle/working/minio.log
8.0K	/kaggle/working/mlflow.log
224K	/kaggle/working/mlflow.db
30M	/kaggle/working/mc
40M	/kaggle/working/images
106M	/kaggle/working/minio
380M	/kaggle/working/repo
745M	/kaggle/working/minio-data


In [8]:
import subprocess, os
subprocess.run(
  "cd /kaggle/working && zip -r ckpt7_light.zip mlflow.db repo/outputs 2>/dev/null",
  shell=True)
print("ckpt7_light.zip:", round(os.path.getsize('/kaggle/working/ckpt7_light.zip')/1e6,2), "МБ")

ckpt7_light.zip: 364.43 МБ


In [10]:
import os, glob, shutil, subprocess
os.environ.update(MLFLOW_TRACKING_URI="http://localhost:5000",
                  MLFLOW_S3_ENDPOINT_URL="http://localhost:9000",
                  AWS_ACCESS_KEY_ID="minioadmin", AWS_SECRET_ACCESS_KEY="minioadmin")
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
c = mlflow.tracking.MlflowClient()
exp = c.get_experiment_by_name("skin_lesion_isic2018")
run = c.search_runs(exp.experiment_id, filter_string="tags.mlflow.runName='vit_b16_final'")[0]

# (A) модель для разраба
m = mlflow.artifacts.download_artifacts(run_id=run.info.run_id,
        artifact_path="service_checkpoint", dst_path="/kaggle/working/_dl")
shutil.copy(glob.glob(m+"/*.pth")[0], "/kaggle/working/model_prd.pth")
print("МОДЕЛЬ:", round(os.path.getsize('/kaggle/working/model_prd.pth')/1e6,1), "МБ")

# (B) графики для презы — стащим все png из артефактов рана
allart = mlflow.artifacts.download_artifacts(run_id=run.info.run_id, dst_path="/kaggle/working/_art")
os.makedirs("/kaggle/working/plots_for_slides", exist_ok=True)
for p in glob.glob(allart+"/**/*.png", recursive=True):
    shutil.copy(p, "/kaggle/working/plots_for_slides/")
for p in glob.glob(allart+"/**/*.csv", recursive=True):
    shutil.copy(p, "/kaggle/working/plots_for_slides/")
print("ГРАФИКИ:", os.listdir("/kaggle/working/plots_for_slides"))

# (C) база MLflow для скриншотов UI
subprocess.run("cd /kaggle/working && zip -q plots_and_db.zip mlflow.db -r plots_for_slides", shell=True)
print("готово: model_prd.pth + plots_and_db.zip")

МОДЕЛЬ: 343.3 МБ


ГРАФИКИ: ['error_examples.png', 'error_examples.csv', 'mel_threshold_tradeoff.png', 'learning_curves.png', 'confusion_matrix.png', 'confused_pairs.csv', 'mel_threshold_compare.csv']
готово: model_prd.pth + plots_and_db.zip
